# Test Case 5 — ECCS Train Degradation: Common-Cause Failure

## Purpose

Demonstrates the **CCF structural delta** scoring (Item 3 from the RCA metamodel) and the
`vendor_supply_chain_records` input pathway on a realistic dual-train ECCS surveillance failure.

## Scenario Summary

| Field | Value |
|---|---|
| **Event ID** | `E2026-02-14-001` |
| **System** | U3 HPCI (High Pressure Coolant Injection) |
| **Failure mode** | Coupling slippage under load — identical on both trains |
| **CCF coupling** | Same vendor lot `VEN-2026-Q1-HC7`, same crew, undispositioned advisory |
| **Primary hypothesis** | `FM-HPCI-CCF-COUPLING` (Category C) |

## Show-stopper

The CCF structural delta gives the CCF candidate a scoring boost from `common_cause_score`
even before evidence refinement. The sensitivity table shows that if the vendor advisory had
been dispositioned into a station WO, CCF confidence would increase further — a direct pointer
to a programmatic gap that a manager can act on immediately.

In [ ]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR   = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR    = NOTEBOOK_ROOT / "rca_runs_case_005"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [
    os.path.abspath(os.path.join(os.getcwd(), "..", ".."))        ,  # RCA root
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")) ,  # dackar root
    os.path.abspath(os.path.join(os.getcwd(), "..", "shared"))    ,  # shared helpers
]:
    if p not in sys.path:
        sys.path.insert(0, p)

from run_helpers import build_fixture_orchestrator, load_fixtures, run_rca, summarise_result
from assertion_helpers import (
    assert_candidate_present, assert_composite_score_above,
    assert_data_coverage_status, assert_barrier_analysis_present,
    assert_degraded_barrier_count, assert_ap913_completeness_present,
    run_assertion_table,
)
print("Imports OK. Fixture dir:", FIXTURE_DIR)

In [ ]:
fixtures = load_fixtures(FIXTURE_DIR)
assert fixtures["event"]["event_id"] == fixtures["telemetry_summary"]["event_id"]
print("Fixture sanity checks passed.")
print("Vendor records loaded:", fixtures["vendor_supply_chain_records"] is not None)

In [ ]:
orchestrator = build_fixture_orchestrator(OUTPUT_DIR, top_k_candidates=5, enable_ishikawa=True)
result = run_rca(orchestrator, fixtures)
print("Pipeline complete.")
summarise_result(result)

In [ ]:
## Inspect CCF scoring
candidates = (result.get("causality_candidates") or {}).get("candidates") or []
for cand in sorted(candidates, key=lambda c: -(c.get("scores") or {}).get("composite", 0)):
    fm_id  = cand.get("failure_mode_id", "?")
    scores = cand.get("scores") or {}
    print(f"  [{fm_id}]")
    print(f"    composite   : {scores.get('composite', '?')}")
    print(f"    ccf_score   : {scores.get('ccf_score', 'n/a')}")
    print(f"    structural  : {scores.get('structural', '?')}")

## Show sensitivity table
sens = (result.get("run_manifest") or {}).get("artifacts", {}).get("sensitivity_table") or []
print(f"\nSensitivity table rows: {len(sens)}")
for row in sens[:5]:
    print(json.dumps(row, indent=2, default=str)[:400])

In [ ]:
assertions = [
    {"id": "A5-1", "desc": "CCF candidate present",
     "fn": lambda r: assert_candidate_present(r, "FM-HPCI-CCF-COUPLING")},
    {"id": "A5-2", "desc": "Vendor supply chain present in coverage",
     "fn": lambda r: assert_data_coverage_status(r, "vendor_supply_chain", "present")},
    {"id": "A5-3", "desc": "Barrier analysis present",
     "fn": lambda r: assert_barrier_analysis_present(r)},
    {"id": "A5-4", "desc": "AP-913 completeness present",
     "fn": lambda r: assert_ap913_completeness_present(r)},
]
run_assertion_table(result, assertions, label="TC-5 Assertions")

In [ ]:
out_path = OUTPUT_DIR / "tc5_full_result.json"
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(result, fh, indent=2, default=str)
print(f"Saved: {out_path}")